# Differential Expression (pseudobulk DESeq2) - headless

Reproduces a paper's DEG finding from nf-core/scrnaseq's per-sample cell-called matrices.

Each mounted h5ad holds every cell of ONE sample. Summing its raw counts per gene gives one
column, and cbind-ing the columns gives the genes x samples integer matrix DESeq2 consumes. The
statistical core below is the same body as `de_bulk_deseq2.ipynb`; only the read-and-aggregate
head is new.


In [ ]:
# Parameters (injected at launch). All are str/number so injection stays valid R.
counts_paths <- "/data/sample_filtered_matrix.h5ad"   # comma-separated per-sample CELL-CALLED h5ad
                                              # files (filtered / cellbender_filter). NEVER `raw`:
                                              # that is every barcode the sequencer saw, mostly
                                              # empty droplets holding ambient RNA.
output_path <- "/outputs/de_results.csv"      # named to match the bulk template: the driver picks
                                              # the session's result table by scoring its NAME.
test_samples <- ""                            # comma-separated sample names (treatment)
reference_samples <- ""                       # comma-separated sample names (control)
block_labels <- ""                            # optional matched-pairs: per-sample block/subject
                                              # labels, comma-separated, ALIGNED to c(test,reference)
lfc_threshold <- 1.0
padj_threshold <- 0.05
gene_id_namespace <- "symbol"                 # "symbol" (var_names, the scanpy default the pipeline
                                              # writes) or "ensembl" (rowData$gene_ids)


In [ ]:
suppressMessages({
  library(zellkonverter)
  library(SingleCellExperiment)
  library(SummarizedExperiment)
  library(Matrix)
})

paths <- trimws(strsplit(counts_paths, ",")[[1]]); paths <- paths[paths != ""]
if (length(paths) == 0) stop("no per-sample h5ad inputs were mounted")

# Pseudobulk: sum RAW integer counts per gene across a sample's cells. Raw, not normalized: DESeq2
# requires integer counts and estimates its own size factors, and feeding it normalized values
# invalidates its dispersion model.
columns <- list()
for (p in paths) {
  sce <- readH5AD(p, use_hdf5 = FALSE)

  # `mtx_to_h5ad_star.py` sets adata.obs["sample"] = meta.id at creation, and meta.id is the
  # samplesheet's `sample` column, which bioAF fills from Sample.external_id. So these labels are
  # the same identifiers the differential design's arms name; no mapping layer is needed.
  if (!("sample" %in% colnames(colData(sce)))) {
    stop(paste0("no obs['sample'] in ", p, "; this is not an nf-core/scrnaseq per-sample matrix"))
  }
  observed_in_file <- unique(as.character(colData(sce)$sample))
  if (length(observed_in_file) != 1) {
    stop(paste0("expected one sample per file, but ", p, " carries ", length(observed_in_file),
                ": ", paste(observed_in_file, collapse = ", "),
                ". Point this notebook at the per-sample matrices, not the concatenated one."))
  }
  sample_name <- observed_in_file[1]
  # `columns[[sample_name]] <- ...` overwrites on a repeat, so a duplicate would silently drop one
  # file and pseudobulk the other twice under that name.
  if (sample_name %in% names(columns)) {
    stop(paste0("two mounted files both carry sample '", sample_name,
                "'; the pseudobulk matrix needs exactly one file per sample."))
  }

  assay_name <- if ("counts" %in% assayNames(sce)) "counts" else assayNames(sce)[1]
  totals <- Matrix::rowSums(assay(sce, assay_name))

  # Report the namespace actually EMITTED, not the one requested: asking for ensembl on a matrix
  # with no rowData$gene_ids falls back to symbols, and two correct gene sets in different
  # namespaces overlap by zero, which reads as a scientific divergence while being purely technical.
  use_ensembl <- gene_id_namespace == "ensembl" && "gene_ids" %in% colnames(rowData(sce))
  used_namespace <- if (use_ensembl) "ensembl" else "symbol"
  ids <- if (use_ensembl) as.character(rowData(sce)$gene_ids) else rownames(sce)
  names(totals) <- ids
  columns[[sample_name]] <- totals
  cat("pseudobulked", ncol(sce), "cells of", sample_name, "from", basename(p),
      "(assay", assay_name, ")\n")
}

# Align on the genes every file measured. The files come from one run against one reference, so this
# is normally the whole set; a shrinking intersection is worth seeing rather than silently padding.
common <- Reduce(intersect, lapply(columns, names))
if (length(common) == 0) stop("the mounted matrices share no gene identifiers")
for (nm in names(columns)) {
  if (length(columns[[nm]]) != length(common)) {
    cat("note:", nm, "measured", length(columns[[nm]]), "genes;", length(common), "are shared\n")
  }
}
mat <- do.call(cbind, lapply(columns, function(v) v[common]))
colnames(mat) <- names(columns)
rownames(mat) <- common
cat("pseudobulk matrix:", nrow(mat), "genes x", ncol(mat), "samples; gene id namespace:",
    used_namespace, "\n")
if (gene_id_namespace != used_namespace) {
  cat("note:", gene_id_namespace, "was requested but the matrices carry no such ids; emitted",
      used_namespace, "instead\n")
}


In [ ]:
suppressMessages(library(DESeq2))

test_s <- trimws(strsplit(test_samples, ",")[[1]]); test_s <- test_s[test_s != ""]
ref_s  <- trimws(strsplit(reference_samples, ",")[[1]]); ref_s <- ref_s[ref_s != ""]
stopifnot(length(test_s) > 0, length(ref_s) > 0)
samples <- c(test_s, ref_s)
condition <- factor(c(rep("test", length(test_s)), rep("reference", length(ref_s))), levels = c("reference", "test"))
coldata <- data.frame(condition = condition, row.names = samples)

# A silent subset would fit the model on fewer samples than the design declares and report a verdict
# for a contrast nobody asked for. Name both sides so the mismatch is diagnosable from the log alone.
missing <- setdiff(samples, colnames(mat))
if (length(missing) > 0) {
  stop(paste0("samples not in the pseudobulk matrix. requested: ", paste(samples, collapse = ", "),
              "; observed: ", paste(colnames(mat), collapse = ", "),
              "; missing: ", paste(missing, collapse = ", ")))
}

# Matched-pairs / blocked design: when a block label is supplied for EVERY sample and there are
# >= 2 distinct labels, model `~ block + condition` so donor-to-donor baseline variance is removed
# (sharply raising power for a paired study). Otherwise fall back to the unpaired `~ condition`.
if (!exists("block_labels")) block_labels <- ""   # tolerate an injector/DB row that omits this optional param
block_s <- trimws(strsplit(block_labels, ",")[[1]]); block_s <- block_s[block_s != ""]
use_block <- length(block_s) == length(samples) && length(unique(block_s)) >= 2
if (use_block) {
  coldata$block <- factor(block_s)
  design_formula <- ~ block + condition
} else {
  design_formula <- ~ condition
}

counts <- mat[, samples, drop = FALSE]
counts <- matrix(as.integer(round(as.numeric(counts))), nrow = nrow(counts), dimnames = dimnames(counts))

dds <- DESeqDataSetFromMatrix(countData = counts, colData = coldata, design = design_formula)
dds <- DESeq(dds)
res <- as.data.frame(results(dds, contrast = c("condition", "test", "reference")))
dir.create(dirname(output_path), showWarnings = FALSE, recursive = TRUE)
out <- data.frame(gene_id = rownames(res), log2FoldChange = res$log2FoldChange, padj = res$padj)
write.csv(out, output_path, row.names = FALSE)
cat("wrote", nrow(out), "genes to", output_path, "\n")
if (use_block) cat("design: ~ block + condition (paired,", length(unique(block_s)), "subjects)\n")
